# StandUp4AI: 1000-Video Evaluation

Evaluate F0/spectral features on 1000 StandUp4AI videos.

**Goal:** Prove F0 works across ALL languages (7 languages in StandUp4AI)
**Baseline:** F1=0.51 @ IoU=0.2 (EMNLP 2025)
**Target:** F1 > 0.51

In [ ]:
# Cell 1: Setup paths
BASE = "/content/drive/MyDrive/standup4ai_1000"
AUDIO_DIR = f"{BASE}/audio"
LABELS_DIR = f"{BASE}/labels"
OUTPUT_DIR = f"{BASE}/results_1000"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check what we have
audio_files = [f.replace('.m4a','') for f in os.listdir(AUDIO_DIR) if f.endswith('.m4a')]
label_files = [f.replace('.csv','') for f in os.listdir(LABELS_DIR) if f.endswith('.csv')]
overlap = set(audio_files) & set(label_files)
print(f"Audio files: {len(audio_files)}")
print(f"Label files: {len(label_files)}")
print(f"Have BOTH: {len(overlap)}")

In [ ]:
# Cell 2: Extract features from all videos
import librosa
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def extract_spectral(audio_path, sr=22050):
    """Extract 20-dim spectral features."""
    try:
        y, sr = librosa.load(audio_path, sr=sr, mono=True)
        rms = librosa.feature.rms(y=y)[0]
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        spec_roll = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
        spec_flat = librosa.feature.spectral_flatness(y=y)[0]
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)[0]
        
        features = []
        for i in range(len(rms)):
            f = [
                rms[i], zcr[i], spec_cent[i], spec_bw[i],
                spec_roll[i], spec_flat[i],
            ] + mfcc[i].tolist()
            features.append(f)
        return np.array(features)
    except:
        return None

# Process all videos
all_features = []
all_labels = []
all_vids = []

for vid in tqdm(overlap):
    audio_path = f"{AUDIO_DIR}/{vid}.m4a"
    label_path = f"{LABELS_DIR}/{vid}.csv"
    
    # Extract features
    feats = extract_spectral(audio_path)
    if feats is None:
        continue
    
    # Load labels
    labels_df = pd.read_csv(label_path)
    labels = (labels_df['label'] == 'risa').astype(int).values
    
    # Match segments (approximate)
    n_segments = min(len(feats), len(labels))
    if n_segments > 0:
        all_features.append(feats[:n_segments])
        all_labels.append(labels[:n_segments])
        all_vids.extend([vid] * n_segments)

X = np.vstack(all_features)
y = np.concatenate(all_labels)
print(f"Dataset: {X.shape}, Positive: {y.sum()}/{len(y)} ({y.mean()*100:.1f}%)")

In [ ]:
# Cell 3: Train and evaluate with video-level CV
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score
import json

# Group by video
groups = np.array(all_vids)
unique_vids = np.unique(groups)
print(f"Videos: {len(unique_vids)}")

# 5-fold video-level CV
gkf = GroupKFold(n_splits=5)
f1s = []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    
    # Scale
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    
    # Train
    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(X_tr_s, y_tr)
    
    # Predict
    y_pred = clf.predict(X_te_s)
    f1 = f1_score(y_te, y_pred)
    f1s.append(f1)
    print(f"Fold {fold+1}: F1={f1:.4f}")

print(f"\nMean F1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"Baseline: F1=0.51 @ IoU=0.2")
print(f"Improvement: {(np.mean(f1s) - 0.51)*100/0.51:.1f}%")

In [ ]:
# Cell 4: Save results
results = {
    'n_videos': len(unique_vids),
    'n_samples': len(y),
    'positive_rate': float(y.mean()),
    'f1_mean': float(np.mean(f1s)),
    'f1_std': float(np.std(f1s)),
    'fold_f1s': [float(f) for f in f1s],
    'baseline_f1': 0.51
}

with open(f"{OUTPUT_DIR}/results_1000.json", 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {OUTPUT_DIR}/results_1000.json")
print(json.dumps(results, indent=2))